# Laminar Backward-Facing-Step with XNSE-Solver
- Test case by Armaly

In [ ]:
#r "BoSSSpad/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using BoSSS.Solution.XdgTimestepping;

In [ ]:
BoSSSshell.WorkflowMgm.Init("BackwardFacingStep");

In [ ]:
var myBatch = ExecutionQueues[1];
myBatch

RuntimeLocation,AdditionalEnvironmentVars,DeploymentBaseDirectory,DeployRuntime,Name,DotnetRuntime,Username,NumOfAdditionalServiceCores,NumOfAdditionalServiceCoresMPISerial,NumOfServiceCoresPerMPIprocess,ServerName,ComputeNodes,DefaultJobPriority,SingleNode,AllowedDatabasesPaths
win\amd64,[ ],\\dc3\userspace\smuda\hpccluster\binaries,True,FDY-WindowsHPC,dotnet,FDY\smuda,0,0,0,DC3,<null>,Normal,True,[ \\dc3\userspace\smuda\hpccluster\ ]


## Problem settings

In [ ]:
double density = 100.0;
double viscosity = 1.0;
double averBulkVelocity = 1.0;
double H = 1.0;

double Re = density * averBulkVelocity * H / viscosity;
Re

100

In [ ]:
double ExpansionRatio = 1.9423;
double h = H / ExpansionRatio;
h

0.514853524172373

In [ ]:
double S = H - h;
S

0.48514647582762704

In [ ]:
Formula InletVelocity = new Formula(
    "VelX",
    false,
    "double VelX(double[] X) { " +
    "double S = 0.48514647582762704;" +
    "double h = 0.514853524172373;" +   
    "double U_max = (3.0/2.0)*1.0;" + 
    "return (-4.0 * U_max / h.Pow2()) * ((X[1] - S) - (h / 2.0)).Pow2() + U_max; } "
);

## Grid Creation

Construct a grid consisting of three parts:
1. Inflow
2. OuterFlow
3. StepFlow

In [ ]:
//int upStreamFactor = 4;
double upStreamLength = H;
double outflowLength = 4.0 * H;
double TotalLength = 5.0 * H;

In [ ]:
int FlowResolution = 4;
int upStreamLengthResolution = 4 * FlowResolution;
int freeStreamHeightResolution = 2 * FlowResolution;
int freeStreamLengthResolution = 16 * FlowResolution;
int stepResolution = 2 * FlowResolution;

In [ ]:
double[] xNodes1 = GenericBlas.Linspace(0, upStreamLength, upStreamLengthResolution + 1);
double[] yNodes1 = GenericBlas.Linspace(S, H, freeStreamHeightResolution + 1);

var grd1 = Grid2D.Cartesian2DGrid(xNodes1, yNodes1);

In [ ]:
double[] xNodes2 = GenericBlas.Linspace(upStreamLength, TotalLength, freeStreamLengthResolution + 1);
double[] yNodes2 = yNodes1; //GenericBlas.Linspace(stepHeight, TotalHeight, 8*freeStreamResolution + 1);

var grd2 = Grid2D.Cartesian2DGrid(xNodes2, yNodes2);

In [ ]:
double[] xNodes3 = xNodes2; //GenericBlas.Linspace(upStreamLength, TotalLength, 20*freeStreamResolution + 1);
double[] yNodes3 = GenericBlas.Linspace(0, S, stepResolution + 1);

var grd3 = Grid2D.Cartesian2DGrid(xNodes3, yNodes3);

In [ ]:
var gridMerged = GridCommons.MergeLogically(new GridCommons[] {grd1, grd2, grd3});
var grd = GridCommons.Seal(gridMerged, 4);

In [ ]:
grd.NumberOfCells

1152

In [ ]:
grd.DefineEdgeTags(delegate(Vector X) {
    string ret = null;
    if (X.x.Abs() <= 1e-8)
        ret = IncompressibleBcType.Velocity_Inlet.ToString();
    if ((X.y - H).Abs() <= 1e-8)
        ret = IncompressibleBcType.Wall.ToString();
    if ((X.x - TotalLength).Abs() <= 1e-8)
        ret = IncompressibleBcType.Pressure_Outlet.ToString();
    if ((X.x < upStreamLength) && (X.y - S).Abs() <= 1e-8)
        ret = IncompressibleBcType.Wall.ToString();
    if ((X.x - upStreamLength).Abs() <= 1e-8 && (X.y < S))
        ret = IncompressibleBcType.Wall.ToString();
    if ((X.x > upStreamLength) && X.y.Abs() <= 1e-8)
        ret = IncompressibleBcType.Wall.ToString();
    return ret;
}); 

## Set up Controls 

In [ ]:
int k = 2;

In [ ]:
List<XNSE_Control> Controls = new List<XNSE_Control>();
Controls.Clear();

In [ ]:
XNSE_Control C = new XNSE_Control();

C.SetGrid(grd);
C.SetDGdegree(k);

C.PhysicalParameters.rho_A = density;
C.PhysicalParameters.mu_A = viscosity;
C.PhysicalParameters.IncludeConvection = true;

C.AddBoundaryValue(IncompressibleBcType.Velocity_Inlet.ToString(), "VelocityX#A", InletVelocity);

C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
C.Option_LevelSetEvolution = LevelSetEvolution.None;

C.SessionName = "BFS_Armaly_SetupTest4";

Controls.Add(C);

## Launch Jobs

In [ ]:
Controls.Select(C => C.SessionName)

index,value
0,BFS_Armaly_SetupTest4


In [ ]:
foreach(var ctrl in Controls) {
    var oneJob              = ctrl.CreateJob();

    oneJob.NumberOfMPIProcs = 1;

    int numThreads = 2;
    oneJob.NumberOfThreads = numThreads;
    IDictionary<string, string> args = oneJob.EnvironmentVars;
    args["BOSSS_ARG_7"] = numThreads.ToString();
    args.Remove("OMP_NUM_THREADS");
    oneJob.MySetCommandLineArguments(args.Values.ToArray());

    oneJob.Activate(myBatch); 
}

## Postprocessing

In [ ]:
wmg.Sessions

#0: BackwardFacingStep	BFS_Armaly_SetupTest4	01/08/2025 09:22:28	7ef65635...
#1: BackwardFacingStep	BFS_Armaly_SetupTest3	01/08/2025 09:13:39	8356a737...
#2: BackwardFacingStep	BFS_Armaly_SetupTest2	01/08/2025 09:00:13	fc5c9a6c...
#3: BackwardFacingStep	BFS_Armaly_SetupTest	01/06/2025 17:37:38	b080c45a...
#4: BackwardFacingStep	BFS_SetupTest4	01/06/2025 14:44:17	fee4f015...
#5: BackwardFacingStep	BFS_SetupTest3	01/06/2025 14:34:06	0bf96d47...
#6: BackwardFacingStep	BFS_SetupTest2	01/06/2025 14:25:14	6ccf71c5...
#7: BackwardFacingStep	BFS_SetupTest	01/06/2025 11:25:37	3a26d035...


In [ ]:
var sess = wmg.Sessions.Pick(0);
sess

BackwardFacingStep	BFS_Armaly_SetupTest4	01/08/2025 09:22:28	7ef65635...

In [ ]:
sess.Export().WithSupersampling(2).Do();